# Qutrit GST

Two ions in one trap, read out by collecting fluorescence from both at once, are not a two-qubit system. The detector counts photons; it cannot say which ion emitted them. Three outcomes come back: no ion bright, one bright, both bright. The states that produce them span the symmetric subspace of the two-qubit Hilbert space, which is three-dimensional, so the thing to characterize is a qutrit, and the third level is a third of the system rather than somewhere population escapes to.

In [1]:
import numpy as np
from numpy import pi

import pygsti
from pygsti.models import qutrit
from pygsti.algorithms.fiducialselection import find_fiducials
from pygsti.algorithms.germselection import find_germs
from pygsti.protocols import ProtocolData, StandardGST, StandardGSTDesign

## The target model

`create_qutrit_model` builds a target model for this situation: four operations on a single line label `T0`, three of them two-qubit unitaries projected onto the symmetric subspace. `Gx` is $X(\theta) \otimes X(\theta)$ and `Gy` is $Y(\theta) \otimes Y(\theta)$, one single-ion rotation applied identically to both ions at once. Applying the *same* rotation to both is what keeps the state inside the symmetric subspace, since any $U \otimes U$ commutes with the swap. `Gm` is the Mølmer-Sørensen unitary $\exp(-i \theta\, A \otimes A / 2)$ with $A = \cos\phi\, \sigma_x + \sin\phi\, \sigma_y$, which at `ms_local=0` reduces to $\exp(-i \theta\, \sigma_x \otimes \sigma_x / 2)$. `Gi` is the identity.

State preparation is $|00\rangle$, and the three POVM effects project onto $|00\rangle$, the symmetric combination of $|01\rangle$ and $|10\rangle$, and $|11\rangle$. Level $i$ is therefore $i$ bright ions, as the effect labels below show.

`basis="qt"` selects pyGSTi's qutrit basis, whose nine elements are two-qubit Pauli products projected onto that same symmetric subspace and orthonormalized under the trace inner product. The operator basis is thus built the way the gates are, and its labels are still written in two-qubit Pauli terms.

In [2]:
target_model = qutrit.create_qutrit_model(error_scale=0, x_angle=pi/2, y_angle=pi/2,
                                          ms_global=pi/2, ms_local=0, basis="qt")

print("state space:", target_model.state_space)
print("operations: ", list(target_model.operations.keys()))
print("effects:    ", list(target_model.povms['Mdefault'].keys()))
print("basis:      ", target_model.basis.labels)

state space: T0(9)
operations:  [Label(('Gi', 'T0')), Label(('Gx', 'T0')), Label(('Gy', 'T0')), Label(('Gm', 'T0'))]
effects:     ['0bright', '1bright', '2bright']
basis:       ['II', 'X+Y', 'X-Y', 'YZ', 'IX', 'IY', 'IZ', 'XY', 'XZ']


## Fiducials and germs

None of pyGSTi's current modelpacks describes a qutrit (the one that does, `stdQT_XYIMS`, sits in the deprecated `legacy` package), so this page searches for fiducials and germs rather than importing them. Both searches finish well under a second here. The candidate lists are all circuits up to a fixed length over the gate alphabet, so they grow exponentially in that length.

In [3]:
fiducialPrep, fiducialMeasure = find_fiducials(target_model, candidate_fid_counts={4: 'all upto'},
                                               algorithm='greedy')
germs = find_germs(target_model, randomize=False, candidate_germ_counts={4: 'all upto'},
                   mode='compactEVD', float_type=np.double)

Initial Length Available Fiducial List: 121
Length Available Fiducial List Dropped Identities and Duplicates: 50
Using greedy algorithm.
Complete initial fiducial set succeeds.
Now searching for best fiducial set.
Starting fiducial list optimization. Lower score is better.
Acceptable candidate solution found.
Score: major=-9 minor=18.0, N: 9
Exiting greedy search.
Preparation fiducials:
['{}@(T0)', 'Gm:T0Gm:T0@(T0)', 'Gx:T0@(T0)', 'Gy:T0@(T0)', 'Gm:T0Gm:T0Gx:T0@(T0)', 'Gy:T0Gy:T0Gy:T0@(T0)', 'Gm:T0@(T0)', 'Gx:T0Gm:T0@(T0)', 'Gm:T0Gy:T0@(T0)']
Score: 18.0
Complete initial fiducial set succeeds.
Now searching for best fiducial set.
Starting fiducial list optimization. Lower score is better.
Acceptable candidate solution found.
Score: major=-9 minor=7.971794871794872, N: 9
Exiting greedy search.
Measurement fiducials:
['{}@(T0)', 'Gx:T0@(T0)', 'Gm:T0@(T0)', 'Gx:T0Gm:T0@(T0)', 'Gy:T0@(T0)', 'Gy:T0Gm:T0@(T0)']
Score: 7.971794871794872
Initial Length Available Germ List: 90
Length Available 

In [4]:
print("%d prep fiducials" % len(fiducialPrep))
print("%d meas fiducials" % len(fiducialMeasure))
print("%d germs" % len(germs))

9 prep fiducials
6 meas fiducials
9 germs


## Circuits and data

`StandardGSTDesign` assembles the germ-power circuits sandwiched between fiducials. The dimension travels with the processor specification, which `create_processor_spec` reads off the model as one qudit of dimension three named `T0`.

In [5]:
maxLengths = [1, 2, 4]
design = StandardGSTDesign(target_model.create_processor_spec(), fiducialPrep, fiducialMeasure,
                           germs, maxLengths)
print("%d circuits" % len(design.all_circuits_needing_data))

709 circuits


Those circuits are what an experiment would have to run. Written out as an empty dataset they become a template: one row per circuit, three count columns to fill in.

In [6]:
pygsti.io.write_empty_dataset("../../../example_files/dataTemplate_qutrit_maxL=4.txt",
                              design.all_circuits_needing_data,
                              "## Columns = 0bright count, 1bright count, 2bright count")

For a real experiment that template is the stopping point: take the data, fill the columns, and read the file back with `pygsti.io.load_dataset`. The rest of this page runs on simulated counts instead, drawn from a depolarized copy of the target model.

In [7]:
mdl_datagen = target_model.depolarize(op_noise=0.05, spam_noise=0.01)
DS = pygsti.data.simulate_data(mdl_datagen, design.all_circuits_needing_data,
                               num_samples=1000, sample_error='multinomial', seed=2018)
data = ProtocolData(design, DS)

## Running GST

`StandardGST` in `CPTPLND` mode fits a Lindblad-parameterized CPTP model. Nothing in the call is qutrit-specific; the three levels arrive with the model and the design.

The `optimizer` argument caps the Levenberg-Marquardt iterations at 50 to keep this page quick. Every stage then stops on the cap rather than on its own convergence test, and says so in the output below. Drop the cap when the answer matters.

In [8]:
result = StandardGST(modes=('CPTPLND',), target_model=target_model,
                     optimizer={'maxiter': 50}, verbosity=4).run(data)

-- Std Practice:  Iter 1 of 1  (CPTPLND) --: 
    Precomputing CircuitOutcomeProbabilityArray layouts for each iteration.
      Using MapForwardSimulator without MPI
      Using MapForwardSimulator without MPI
      Using MapForwardSimulator without MPI
  --- Iterative GST: Iter 1 of 3  181 circuits ---: 
    --- chi2 GST ---
      --- Outer Iter 0: norm_f = 4.20034e+06, mu=1, |x|=0, |J|=1850.62
      --- Outer Iter 1: norm_f = 369684, mu=80.1269, |x|=0.0838399, |J|=20689.7
      --- Outer Iter 2: norm_f = 29733.4, mu=31.9548, |x|=1.04598, |J|=1422.25
      --- Outer Iter 3: norm_f = 12637.1, mu=31.6697, |x|=1.36627, |J|=1090.22
      --- Outer Iter 4: norm_f = 2837.28, mu=23.9477, |x|=1.03939, |J|=1365.98
      --- Outer Iter 5: norm_f = 1926.06, mu=49.3526, |x|=0.990365, |J|=1419.71
      --- Outer Iter 6: norm_f = 1155.26, mu=49.4439, |x|=0.959402, |J|=1473.31
      --- Outer Iter 7: norm_f = 905.876, mu=55.5699, |x|=0.942316, |J|=1489.79
      --- Outer Iter 8: norm_f = 820.868, mu

      --- Outer Iter 0: norm_f = 738.358, mu=1, |x|=0.821306, |J|=2229.5
      --- Outer Iter 1: norm_f = 607.638, mu=488.029, |x|=0.835965, |J|=2192.61
      --- Outer Iter 2: norm_f = 485.684, mu=162.676, |x|=0.81177, |J|=2230.47
      --- Outer Iter 3: norm_f = 472.108, mu=312.626, |x|=0.811758, |J|=2231.64
      --- Outer Iter 4: norm_f = 461.524, mu=185.826, |x|=0.8083, |J|=2236.02
      --- Outer Iter 5: norm_f = 458.785, mu=185.165, |x|=0.807926, |J|=2233.86
      --- Outer Iter 6: norm_f = 456.209, mu=124.991, |x|=0.804671, |J|=2236.37
      --- Outer Iter 7: norm_f = 455.532, mu=124.993, |x|=0.803861, |J|=2234.8
      --- Outer Iter 8: norm_f = 454.577, mu=107.862, |x|=0.802235, |J|=2235.46
      --- Outer Iter 9: norm_f = 454.08, mu=105.988, |x|=0.801322, |J|=2235.04
      --- Outer Iter 10: norm_f = 453.61, mu=95.8224, |x|=0.800079, |J|=2235.36
      --- Outer Iter 11: norm_f = 453.379, mu=95.8319, |x|=0.799352, |J|=2235.08
      --- Outer Iter 12: norm_f = 453.112, mu=183.6

  --- Iterative GST: Iter 3 of 3  709 circuits ---: 
    --- chi2 GST ---
      --- Outer Iter 0: norm_f = 1714.84, mu=1, |x|=0.792159, |J|=3594.13
      --- Outer Iter 1: norm_f = 1673.29, mu=839.84, |x|=0.808383, |J|=3517.4
      --- Outer Iter 2: norm_f = 1336.66, mu=279.947, |x|=0.778164, |J|=3596.08
      --- Outer Iter 3: norm_f = 1314.28, mu=554.797, |x|=0.777775, |J|=3597.62
      --- Outer Iter 4: norm_f = 1290.46, mu=371.902, |x|=0.774638, |J|=3609.11
      --- Outer Iter 5: norm_f = 1282.98, mu=366.812, |x|=0.774663, |J|=3610.1
      --- Outer Iter 6: norm_f = 1275.97, mu=276.293, |x|=0.773018, |J|=3615.07
      --- Outer Iter 7: norm_f = 1273.52, mu=275.315, |x|=0.773325, |J|=3614.86
      --- Outer Iter 8: norm_f = 1270.96, mu=236.574, |x|=0.773202, |J|=3616.84
      --- Outer Iter 9: norm_f = 1269.71, mu=230.278, |x|=0.773838, |J|=3616.89
      --- Outer Iter 10: norm_f = 1268.59, mu=205.35, |x|=0.774212, |J|=3617.36
      --- Outer Iter 11: norm_f = 1267.82, mu=197.647, 

      --- Outer Iter 0: norm_f = 632.851, mu=1, |x|=0.785256, |J|=2548.82
      --- Outer Iter 1: norm_f = 632.372, mu=100.421, |x|=0.784422, |J|=2551.26
      --- Outer Iter 2: norm_f = 632.354, mu=33.4738, |x|=0.78444, |J|=2551.18
      --- Outer Iter 3: norm_f = 632.348, mu=13.0806, |x|=0.784288, |J|=2551.15
      --- Outer Iter 4: norm_f = 632.346, mu=12.3598, |x|=0.783904, |J|=2551.2
      --- Outer Iter 5: norm_f = 632.343, mu=8.6671, |x|=0.783493, |J|=2551.35
      --- Outer Iter 6: norm_f = 632.342, mu=8.64342, |x|=0.78306, |J|=2551.42
      --- Outer Iter 7: norm_f = 632.34, mu=7.70059, |x|=0.782676, |J|=2551.53
      --- Outer Iter 8: norm_f = 632.338, mu=7.57697, |x|=0.782304, |J|=2551.59
      --- Outer Iter 9: norm_f = 632.337, mu=7.55296, |x|=0.781969, |J|=2551.64
      --- Outer Iter 10: norm_f = 632.337, mu=13.8561, |x|=0.781673, |J|=2551.63
      --- Outer Iter 11: norm_f = 632.336, mu=15.7, |x|=0.781501, |J|=2551.68
      --- Outer Iter 12: norm_f = 632.335, mu=15.566

## The report

The standard report needs nothing qutrit-specific either.

In [9]:
ws = pygsti.report.construct_standard_report(
    result, "Example Qutrit Report", verbosity=3
).write_html('../../../example_files/sampleQutritReport', connected=True, auto_open=False, verbosity=3)

Running idle tomography
Computing switchable properties


<repo>/pygsti/report/workspacetables.py:849: UserWarning: Failed gauge-robust decomposition of Gx:T0 op:
Cannot construct a real log: unpaired negative real eigenvalues: [np.complex128(-0.001555024426083079+1.3676796782023754e-17j)]
  _warnings.warn("Failed gauge-robust decomposition of %s op:\n%s" % (gl, str(e)))
<env>/lib/python3.13/site-packages/plotly/offline/offline.py:150: UserWarning: 
Unrecognized config options supplied: ['showLink', 'linkText']
  warnings.warn(


Served with these docs: <a href="../../../reports/sampleQutritReport.html">sampleQutritReport</a>.